In [1]:
import pandas as pd

# =========================
# CARREGANDO O DATAFRAME
# =========================

df_peixes = pd.read_csv("Registro de Peixes.csv")

# Criando uma cópia para tratamento dos dados

newdf_peixes = df_peixes.copy()

display(newdf_peixes.head())

,Nome,Estacao,Clima,Horario,Localizacao,Preco_Base,Preco_Ouro,Preco_Iridio,Dificuldade,Comportamento,Xp_Pesca,Vara_Recomendada,Isca_Recomendada
0,Baiacu,Verão,Sol,12:00-16:00,Oceano,200,300.0,400.0,80.0,Flutuante,29.0,Vara_de_fibra_de_vidro,Isca_de_Luxo
1,Anchova,Primavera/Outono,Qualquer,Qualquer_Hora,Oceano,30,45.0,60.0,30.0,Dardo,13.0,Vara_de_bambu,Isca
2,Atum,Verão/Inverno,Qualquer,6:00-19:00,Oceano,100,150.0,200.0,70.0,Suave,26.0,Vara_de_fibra_de_vidro,Isca_de_Luxo
3,Sardinha,Primavera/Outono/Inverno,Qualquer,6:00-19:00,Oceano,40,60.0,80.0,30.0,Dardo,13.0,Vara_de_treinamento,Isca
4,Brema,Primavera/Verão/Outono/Inverno,Qualquer,18:00-2:00,Rio_Cidade/Rio_Floresta,45,67.0,90.0,35.0,Suave,14.0,Vara_de_treinamento,Isca


In [2]:
# =========================
# TRATAMENTO DE PREÇOS NULOS
# =========================

# Alguns peixes de covo não possuem qualidade ouro ou irídio.
# Vamos substituir os valores nulos pelo preço base.

newdf_peixes["Preco_Ouro"] = (newdf_peixes["Preco_Ouro"].fillna(newdf_peixes["Preco_Base"]))

newdf_peixes["Preco_Iridio"] = (newdf_peixes["Preco_Iridio"].fillna(newdf_peixes["Preco_Base"]))

In [3]:
# =========================
# TRATAMENTO DE DIFICULDADE
# =========================

# Peixes de covo não possuem mini-game de pesca.
# Vamos preencher valores nulos com 0 inicialmente.

newdf_peixes["Dificuldade"] = (newdf_peixes["Dificuldade"].fillna(0))

In [4]:
# =========================
# TRATAMENTO DE COMPORTAMENTO
# =========================

# Peixes de covo não possuem comportamento.
# Vamos preencher como "Covo".

newdf_peixes["Comportamento"] = (newdf_peixes["Comportamento"].fillna("Covo"))

# Corrigindo células vazias

newdf_peixes["Comportamento"] = (newdf_peixes["Comportamento"].replace(r'^\s*$', "Covo", regex=True))

In [5]:
# =========================
# TRATAMENTO DE XP
# =========================

# Peixes de covo não fornecem XP de pesca.

newdf_peixes["Xp_Pesca"] = (newdf_peixes["Xp_Pesca"].fillna(0))

In [6]:
# =========================
# TRATAMENTO DE HORÁRIO
# =========================

# Alguns peixes podem ser pescados a qualquer hora do dia.
# O jogo funciona das 00:00 até 26:00.

newdf_peixes["Horario"] = (newdf_peixes["Horario"].replace("Qualquer_Hora", "00:00-26:00"))

# Removendo espaços extras

newdf_peixes["Horario"] = (newdf_peixes["Horario"].str.replace(" ", "", regex=False))

# Alguns peixes possuem múltiplos horários separados por "/".
# Vamos manter apenas o primeiro intervalo para simplificar a análise.

newdf_peixes["Horario"] = (newdf_peixes["Horario"].str.split("/").str[0])

# Separando horário de início e fim

newdf_peixes[["Horario_Inicio", "Horario_Fim"]] = (newdf_peixes["Horario"].str.split("-", expand=True))

In [7]:
# =========================
# ORGANIZANDO COLUNAS
# =========================

newdf_peixes = newdf_peixes[
    [
        "Nome",
        "Estacao",
        "Clima",
        "Horario",
        "Horario_Inicio",
        "Horario_Fim",
        "Localizacao",
        "Preco_Base",
        "Preco_Ouro",
        "Preco_Iridio",
        "Dificuldade",
        "Comportamento",
        "Xp_Pesca",
        "Vara_Recomendada",
        "Isca_Recomendada"
    ]
]


In [8]:
# =========================
# AJUSTE DE DIFICULDADE DOS PEIXES DE COVO
# =========================

# Mesmo sem mini-game, eles ainda possuem tempo de espera e coleta.
# Vamos definir uma dificuldade mínima para balancear melhor a análise.

newdf_peixes.loc[newdf_peixes["Comportamento"] == "Covo","Dificuldade"] = 10

In [9]:
# =========================
# CRIANDO COLUNA DE CUSTO BENEFÍCIO
# =========================

# Quanto maior o valor:
# - mais lucrativo o peixe tende a ser
# - mais fácil ele tende a ser
#
# A dificuldade possui peso maior na fórmula.

newdf_peixes["Custo_Beneficio"] = (newdf_peixes["Preco_Base"] /((newdf_peixes["Dificuldade"] + 1) ** 2))

In [10]:
# =========================
# PEIXES MAIS LUCRATIVOS DO OCEANO
# =========================

peixes_oceano = newdf_peixes[newdf_peixes["Localizacao"].str.contains("Oceano", na=False)]

peixes_oceano_lucrativos = (peixes_oceano[["Nome", "Preco_Base", "Dificuldade", "Custo_Beneficio"]].sort_values(by="Custo_Beneficio", ascending=False))

display(peixes_oceano_lucrativos.head(10))

# =========================
# DATAFRAME FINAL
# =========================

display(newdf_peixes.head(5))

,Nome,Preco_Base,Dificuldade,Custo_Beneficio
61,Lagosta,120,10.0,0.991736
63,Caranguejo,100,10.0,0.826446
68,Lagostim,75,10.0,0.619835
69,Lesma,65,10.0,0.537190
66,Camarão,60,10.0,0.495868
62,Concha,50,10.0,0.413223
64,Berbigão,50,10.0,0.413223
67,Ostra,40,10.0,0.330579
65,Mexilhão,30,10.0,0.247934
70,Caramujo,20,10.0,0.165289


,Nome,Estacao,Clima,Horario,Horario_Inicio,Horario_Fim,Localizacao,Preco_Base,Preco_Ouro,Preco_Iridio,Dificuldade,Comportamento,Xp_Pesca,Vara_Recomendada,Isca_Recomendada,Custo_Beneficio
0,Baiacu,Verão,Sol,12:00-16:00,12:00,16:00,Oceano,200,300.0,400.0,80.0,Flutuante,29.0,Vara_de_fibra_de_vidro,Isca_de_Luxo,0.030483
1,Anchova,Primavera/Outono,Qualquer,00:00-26:00,00:00,26:00,Oceano,30,45.0,60.0,30.0,Dardo,13.0,Vara_de_bambu,Isca,0.031217
2,Atum,Verão/Inverno,Qualquer,6:00-19:00,6:00,19:00,Oceano,100,150.0,200.0,70.0,Suave,26.0,Vara_de_fibra_de_vidro,Isca_de_Luxo,0.019837
3,Sardinha,Primavera/Outono/Inverno,Qualquer,6:00-19:00,6:00,19:00,Oceano,40,60.0,80.0,30.0,Dardo,13.0,Vara_de_treinamento,Isca,0.041623
4,Brema,Primavera/Verão/Outono/Inverno,Qualquer,18:00-2:00,18:00,2:00,Rio_Cidade/Rio_Floresta,45,67.0,90.0,35.0,Suave,14.0,Vara_de_treinamento,Isca,0.034722


In [11]:
newdf_peixes.to_csv("Registro_de_Peixes_Tratado.csv",index=False)